In [1]:
%load_ext autoreload
%autoreload 2
from seek_code import SEEK
from data_handler import EleHandler
from seek_code import SEEK
from concept_head import ConceptHead

from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler

import torch
import torch.nn as nn
import torch.optim as optim

/data/vision/beery/scratch/antoine/micromamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
dataset = EleHandler( )


# Filter the dataset for elephants that have the season 'EFE_IDI'
filtered_indices = [i for i, (_, _, _, identified, _, _, _, _, _, _) in enumerate(dataset) if identified]

print(f"Filtered dataset has {len(filtered_indices)} samples")

# Split the filtered dataset along encounters
train_indices, val_indices, test_indices = dataset.split_along_encounters()

filtered_train_indices = [i for i in train_indices if i in filtered_indices]
filtered_val_indices = [i for i in val_indices if i in filtered_indices]
filtered_test_indices = [i for i in test_indices if i in filtered_indices]

print(f"Train: {len(filtered_train_indices)}, Val: {len(filtered_val_indices)}, Test: {len(filtered_test_indices)}")


Filtered dataset has 5348 samples
Train indices: 6108, Validation indices: 1101, Test indices: 1235
Train: 3940, Val: 626, Test: 782


In [19]:

# Create DataLoaders for the filtered dataset
filtered_train_loader, filtered_val_loader, filtered_test_loader = (
    DataLoader(dataset, batch_size=128, sampler=RandomSampler(indices), num_workers=4) 
    for indices in [filtered_train_indices, filtered_val_indices, filtered_test_indices]
)


In [20]:
ch = ConceptHead()
ch.test(filtered_test_loader)

# Would be nice to make this a method in the ConceptHead class called (freeze everything)
ch.layer.eval()
for param in ch.layer.parameters():
    param.requires_grad = False

ch.backbone.eval()
for param in ch.backbone.parameters():
    param.requires_grad = False

classifier_layer = nn.Sequential(
    nn.Linear(2304, 1017),
    nn.Sigmoid()
)

classifier_layer = classifier_layer.to('cuda' )
criterion = nn.CrossEntropyLoss()  # For multiclass classification
optimizer = optim.Adam(classifier_layer.parameters(), lr=0.001)

/data/vision/beery/scratch/antoine/CBM_reid/methods/concept_head.py:40: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(Path(__file__).parent.parent / 

Test Loss: 0.0691
sex Accuracy: 96.88%
age Accuracy: 97.10%
right_tusk Accuracy: 85.79%
left_tusk Accuracy: 89.49%
R_tear_1 Accuracy: 65.90%
R_hole_1 Accuracy: 78.17%
R_tear_2 Accuracy: 73.36%
R_hole_2 Accuracy: 84.66%
L_tear_1 Accuracy: 62.04%
L_hole_1 Accuracy: 76.39%
L_tear_2 Accuracy: 67.20%
L_hole_2 Accuracy: 83.67%
right_extreme Accuracy: 87.80%
left_extreme Accuracy: 82.56%
ear_special Accuracy: 89.25%
body_special Accuracy: 94.42%
average Accuracy: 82.17%
whole_code Accuracy: 14.76%


In [21]:
def compute_accuracy(logits, labels):
    """Compute accuracy given logits and true labels."""
    _, predicted = torch.max(logits, dim=1)  # Get predicted classes
    correct = (predicted == labels).sum().item()
    total = labels.size(0)
    accuracy = correct / total
    return accuracy * 100  # Return percentage


In [ ]:
def epoch_pass(loader, training = True):
    
    classifier_layer.train() if training else classifier_layer.eval()
    total_loss, total_accuracy, total_batches = 0,0,0

    for batch in filtered_train_loader:
        images, ele_id_label, subject_SEEK, left_ears, right_ears = batch[0].to('cuda'),  batch[2].to('cuda'), batch[4], batch[6].to('cuda'), batch[7].to('cuda')

        with torch.no_grad():
            embeddings = ch.backbone(images)
            if ch.crop_ears:
                left_embeddings = ch.backbone(left_ears)
                right_embeddings = ch.backbone(right_ears)
                embeddings = torch.cat((embeddings, left_embeddings, right_embeddings), dim=1)
        
        # Forward pass
        logits = classifier_layer(embeddings)
        loss = criterion(logits, ele_id_label)  # Loss with integer labels
        total_loss += loss.item()

        # Compute metrics
        total_accuracy += compute_accuracy(logits, ele_id_label)
        total_batches += 1

        # Backward pass and optimization
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    epoch_loss = total_loss / len(filtered_train_loader)
    epoch_accuracy = total_accuracy / total_batches

    return epoch_loss, epoch_accuracy

In [ ]:
for epoch in range(10):
    train_loss, train_accuracy = epoch_pass(filtered_train_loader, training=True)
    val_loss, val_accuracy = epoch_pass(filtered_val_loader, training=False)

    print(f"Epoch {epoch + 1}/{10} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

test_loss, test_accuracy = epoch_pass(filtered_test_loader, training=False)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Epoch 1/10 - Train Loss: 6.8106, Train Accuracy: 0.69% - Val Loss: 6.5863, Val Accuracy: 7.11%
Epoch 2/10 - Train Loss: 6.5625, Train Accuracy: 8.39% - Val Loss: 6.4740, Val Accuracy: 13.30%
Epoch 3/10 - Train Loss: 6.4604, Train Accuracy: 14.00% - Val Loss: 6.4103, Val Accuracy: 15.38%
Epoch 4/10 - Train Loss: 6.4001, Train Accuracy: 14.26% - Val Loss: 6.3618, Val Accuracy: 15.39%
Epoch 5/10 - Train Loss: 6.3576, Train Accuracy: 15.05% - Val Loss: 6.3297, Val Accuracy: 16.82%
Epoch 6/10 - Train Loss: 6.3274, Train Accuracy: 16.67% - Val Loss: 6.3074, Val Accuracy: 17.23%
Epoch 7/10 - Train Loss: 6.3057, Train Accuracy: 17.00% - Val Loss: 6.2864, Val Accuracy: 18.09%
Epoch 8/10 - Train Loss: 6.2853, Train Accuracy: 18.44% - Val Loss: 6.2705, Val Accuracy: 20.37%
Epoch 9/10 - Train Loss: 6.2675, Train Accuracy: 20.88% - Val Loss: 6.2565, Val Accuracy: 21.94%
Epoch 10/10 - Train Loss: 6.2544, Train Accuracy: 21.96% - Val Loss: 6.2437, Val Accuracy: 21.86%


In [26]:
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 6.2441, Test Accuracy: 21.88%


# Training a projector

In [29]:
# Freeze the classifier layer
classifier_layer.eval()
for param in classifier_layer.parameters():
    param.requires_grad = False

In [ ]:
def intervene(predicted_concepts, true_concepts, correction_probability = 0.5):
    for i in range(predicted_concepts.shape[0]):
        if torch.rand(1).item() < correction_probability:
            predicted_concepts[i] = true_concepts[i]
    # To-do correction
    return predicted_concepts


In [39]:
projector_1 = nn.Sequential(
    nn.Linear(63, 512),
    nn.LeakyReLU(),
    nn.Linear(512, 2304),
    nn.LeakyReLU()
)

projector_1 = projector_1.to('cuda')
criterion = nn.CrossEntropyLoss()  # For multiclass classification
optimizer = optim.Adam(projector_1.parameters(), lr=0.001)

In [ ]:
def epoch_pass(loader, training=True, intervention = 0):
    # Freeze Backbone
    ch.backbone.eval()
    for param in ch.backbone.parameters(): param.requires_grad = False

    #Freeze the concept head layer
    ch.layer.eval()
    for param in ch.layer.parameters(): param.requires_grad = False

    # Freeze the classifier layer
    classifier_layer.eval()
    for param in classifier_layer.parameters(): param.requires_grad = False

    # Are we training ?
    projector_1.train() if training else projector_1.eval()

    total_loss, total_accuracy, total_batches = 0, 0, 0

    for batch in loader:
        images, ele_id_label, subject_SEEK, left_ears, right_ears = batch[0].to('cuda'),  batch[2].to('cuda'), batch[4], batch[6].to('cuda'), batch[7].to('cuda')

        with torch.no_grad():
            embeddings = ch.backbone(images)
            if ch.crop_ears:
                left_embeddings = ch.backbone(left_ears)
                right_embeddings = ch.backbone(right_ears)
                embeddings = torch.cat((embeddings, left_embeddings, right_embeddings), dim=1)

            # Compute the concepts from the backbone embeddings
            concept_logits = ch.layer(embeddings)
            predicted_concepts = SEEK.closest_valid_one_hot(concept_logits)
            
            # Allow for intervention
            edited_concepts = intervene(predicted_concepts, subject_SEEK, intervention)
        
        # Projecting the concepts back to the embeddings space
        projected_concepts = projector_1(edited_concepts)
        # I want to do this > loss = criterion(projected_concepts, embeddings)
        
        # We add the concepts projected back to the intiial embeddings space
        edited_embeddings = projected_concepts + embeddings

        # Now we can pass the edited embeddings to the classifier
        logits = classifier_layer(edited_embeddings)

        # Compute the CE loss
        loss = criterion(logits, ele_id_label)

        # Backward pass and optimization
        if training:
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Compute metrics
        total_accuracy += compute_accuracy(logits, ele_id_label)
        total_batches += 1
        total_loss += loss.item()
    
    epoch_loss = total_loss / len(loader)
    epoch_accuracy = total_accuracy / total_batches

    return epoch_loss, epoch_accuracy
        


In [ ]:
for epoch in range(10):
    train_loss, train_accuracy = epoch_pass(filtered_train_loader, training=True, intervention = 0.0)
    val_loss, val_accuracy = epoch_pass(filtered_val_loader, training=False, intervention = 0.0 )

    print(f"Epoch {epoch + 1}/{10} - Train Loss: {train_loss:.4f}, Train Accuracy: {train_accuracy:.2f}% - Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")


test_loss, test_accuracy = epoch_pass(filtered_test_loader, training=False, intervention = 0.0)
print(f"Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.2f}%")

Epoch 1/10 - Train Loss: 6.2352, Train Accuracy: 21.79% - Val Loss: 6.2163, Val Accuracy: 23.94%
Epoch 2/10 - Train Loss: 6.2346, Train Accuracy: 22.02% - Val Loss: 6.2143, Val Accuracy: 23.70%
Epoch 3/10 - Train Loss: 6.2324, Train Accuracy: 22.16% - Val Loss: 6.2139, Val Accuracy: 24.19%
Epoch 4/10 - Train Loss: 6.2307, Train Accuracy: 22.17% - Val Loss: 6.2121, Val Accuracy: 24.54%
Epoch 5/10 - Train Loss: 6.2285, Train Accuracy: 22.29% - Val Loss: 6.2108, Val Accuracy: 25.71%
Epoch 6/10 - Train Loss: 6.2269, Train Accuracy: 22.38% - Val Loss: 6.2091, Val Accuracy: 24.66%
Epoch 7/10 - Train Loss: 6.2260, Train Accuracy: 22.28% - Val Loss: 6.2085, Val Accuracy: 24.84%
Epoch 8/10 - Train Loss: 6.2245, Train Accuracy: 22.47% - Val Loss: 6.2074, Val Accuracy: 25.54%
Epoch 9/10 - Train Loss: 6.2236, Train Accuracy: 22.55% - Val Loss: 6.2040, Val Accuracy: 25.34%
Epoch 10/10 - Train Loss: 6.2209, Train Accuracy: 22.50% - Val Loss: 6.2044, Val Accuracy: 24.60%
